# Aula 02 — Fundamentos Arquiteturais NoSQL: Teoremas CAP & PACELC e ACID vs BASE
**Disciplina:** Bancos de Dados Não-Relacionais (ADS16)  
**Curso:** Tecnologia em Análise e Desenvolvimento de Sistemas — IFCE Campus Tauá  
**Professor:** Prof. Me. Reginaldo Fernandes  
**Semestre:** 2026.2  

---
## 🎯 Objetivos de Aprendizagem Prática:
1. Simular uma topologia de cluster com 3 nós com injeção de rompimento de rede (*Split-Brain*).
2. Experimentar a diferença entre sistemas **CP** (Consistência Estrita) e **AP** (Alta Disponibilidade com Consistência Eventual).
3. Medir a latência do trade-off do **Teorema PACELC** (Daniel Abadi, 2012).
4. Compreender e configurar **Write Concern** e **Read Concern** no MongoDB.

In [ ]:
# Importação de bibliotecas essenciais para a simulação distribuída
import time
import json
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional

print('✓ Ambiente configurado para simulação distribuída!')

## 1. O Teorema CAP (Eric Brewer, 2000 / 2012)
Em qualquer sistema distribuído de dados, a partição de rede (**P**) é uma fatalidade física (cabos rompidos, falhas de roteador, latência extrema). Quando a partição ocorre, o arquiteto é forçado a escolher:
- **CP (Consistency + Partition Tolerance):** Rejeita leituras/escritas em partições sem quórum para impedir divergência de dados e **gasto duplo** (*Double-Spending*).
- **AP (Availability + Partition Tolerance):** Continua respondendo a qualquer nó, mesmo que isolado, aceitando leituras defasadas (*Stale Reads*) e reconciliando os nós depois via **Consistência Eventual**.

In [ ]:
@dataclass
class RegistroDado:
    chave: str
    valor: Any
    versao: int
    timestamp_ms: float
    no_origem: str

class NoDistribuido:
    def __init__(self, id_no: str):
        self.id_no = id_no
        self.dados: Dict[str, RegistroDado] = {}
        self.relogio = 0

    def gravar_local(self, chave: str, valor: Any, ts: float, origem: str):
        self.relogio += 1
        reg = RegistroDado(chave, valor, self.relogio, ts, origem)
        self.dados[chave] = reg
        return reg

    def ler_local(self, chave: str):
        return self.dados.get(chave)

class ClusterDistribuido:
    def __init__(self):
        self.nos = {
            'No-A': NoDistribuido('No-A (São Paulo)'),
            'No-B': NoDistribuido('No-B (Fortaleza)'),
            'No-C': NoDistribuido('No-C (Tauá)')
        }
        self.links_rompidos = set()

    def isolar_no(self, id_no: str):
        for n in self.nos:
            if n != id_no:
                self.links_rompidos.add((id_no, n))
                self.links_rompidos.add((n, id_no))

    def restaurar_rede(self):
        self.links_rompidos.clear()

    def pode_comunicar(self, n1: str, n2: str) -> bool:
        return (n1, n2) not in self.links_rompidos

    def nos_visiveis(self, origem: str) -> List[str]:
        return [n for n in self.nos if self.pode_comunicar(origem, n)]

print('✓ Cluster de 3 nós instanciado!')

### Executando a Injeção de Falha (Split-Brain) e o Teste de Quórum
Rompemos a comunicação com o **No-C (Tauá)** e tentamos realizar operações simultâneas.

In [ ]:
cluster = ClusterDistribuido()
# Inicializa saldo de R$ 1.000,00 nos 3 nós
t0 = time.time() * 1000
for n in cluster.nos.values():
    n.gravar_local('saldo:CTA-10', 1000.0, t0, 'init')

print('1. Saldo inicial replicado nos 3 nós: R$ 1.000,00')

# Injetando o rompimento de rede
cluster.isolar_no('No-C')
print('2. [FALHA] No-C (Tauá) isolado da rede!')

# 1. Teste CP (Consistência Estrita)
visiveis = cluster.nos_visiveis('No-C')
quorum = len(cluster.nos) // 2 + 1
if len(visiveis) < quorum:
    print(f'   [CP] ERRO: No-C enxerga apenas {len(visiveis)} nó(s). Quórum mínimo = {quorum}.')
    print('   [CP] Ação: Escrita BLOQUEADA para impedir inconsistência financeira!')

# 2. Teste AP (Alta Disponibilidade com Consistência Eventual)
cluster.nos['No-C'].gravar_local('item:carrinho', 'Notebook Dell', time.time() * 1000, 'No-C')
print('   [AP] SUCESSO: No-C aceitou a escrita localmente sem aguardar quórum!')
item_a = cluster.nos['No-A'].ler_local('item:carrinho')
print(f'   [AP] No-A enxerga: {item_a} -> Stale Read (Leitura Defasada)')

# 3. Reconciliação (Cura da Rede)
cluster.restaurar_rede()
rec = cluster.nos['No-C'].ler_local('item:carrinho')
cluster.nos['No-A'].dados['item:carrinho'] = rec
cluster.nos['No-B'].dados['item:carrinho'] = rec
print('3. Rede recuperada e reconciliada via Last-Write-Wins (LWW)!')
print(f'   No-A agora possui: {cluster.nos["No-A"].ler_local("item:carrinho").valor}')

## 2. O Teorema PACELC (Daniel Abadi, 2012)
Em operação normal (sem partições), sistemas distribuídos enfrentam outro trade-off crucial:
$$\text{Se } \mathbf{P} \text{ (Partição)} \implies \mathbf{A} \lor \mathbf{C}; \quad \mathbf{E}\text{lse (Normalidade)} \implies \mathbf{L} \lor \mathbf{C}$$

Vamos medir numericamente o impacto na latência entre uma confirmação local (**EL**) e uma confirmação síncrona com todas as réplicas (**EC**):

In [ ]:
rtt_local = 1.2        # Milissegundos (datacenter local)
rtt_fortaleza = 45.0   # Milissegundos (cross-region)
rtt_taua = 58.0        # Milissegundos (interior)

latencia_ec = max(rtt_local, rtt_fortaleza, rtt_taua) + 2.5
latencia_el = rtt_local + 0.3

print(f'Latência PC/EC (Consistência Síncrona Total)  : {latencia_ec:.1f} ms')
print(f'Latência PA/EL (Baixa Latência Assíncrona W=1) : {latencia_el:.1f} ms')
print(f'Aceleração no modelo PA/EL: {latencia_ec / latencia_el:.1f}x mais veloz!')

## 3. Prática com MongoDB: Níveis de Write Concern
No MongoDB, podemos ajustar a durabilidade e a latência em nível de operação:
```javascript
// 1. Alta Performance / Baixa Latência (PA/EL):
db.sensores.insertOne(
  { sensor: 'S-01', temp: 28.5 },
  { writeConcern: { w: 1 } }
);

// 2. Alta Segurança Bancária e Durabilidade (PC/EC):
db.transferencias.insertOne(
  { de: 'CTA-01', para: 'CTA-02', valor: 2500.00 },
  { writeConcern: { w: 'majority', j: true, wtimeout: 5000 } }
);
```